In [8]:
import numpy as np

seed = 42
d_model = 512

rng = np.random.RandomState(seed)
scale = 1.0 / np.sqrt(d_model)
tokens = ["le", "chat", "noir", "dort", "."]
seq_len = len(tokens)
X = np.random.randn(seq_len, d_model) * 0.5

# One big projection matrix for Q, K, V each — we'll reshape into heads
W_Q = rng.randn(d_model, d_model) * scale
W_K = rng.randn(d_model, d_model) * scale
W_V = rng.randn(d_model, d_model) * scale


In [12]:
import torch
import torch.nn.functional as F

print(f"\n  PyTorch version: {torch.__version__}")

# Same sentence, but as a torch tensor
X_t = torch.tensor(X, dtype=torch.float32)

# Use the built-in F.scaled_dot_product_attention
# We need (batch, heads, seq, dim) shape for it
d_k = d_model

Q_t = X_t.unsqueeze(0).unsqueeze(0)  # (1, 1, seq, d_model)
K_t = X_t.unsqueeze(0).unsqueeze(0)
V_t = X_t.unsqueeze(0).unsqueeze(0)

# Without mask
out_t = F.scaled_dot_product_attention(Q_t, K_t, V_t)
print(f"  Output shape (no mask)     : {tuple(out_t.shape)}")

# With causal mask
out_t_causal = F.scaled_dot_product_attention(Q_t, K_t, V_t, is_causal=True)
print(f"  Output shape (causal mask) : {tuple(out_t_causal.shape)}")

# Manual implementation for pedagogy
def attention_torch(Q, K, V, mask=None):
    """Q, K, V: (seq, d_k). Returns (seq, d_v), (seq, seq)."""
    d_k = Q.shape[-1]
    scores = Q @ K.T / (d_k ** 0.5)
    if mask is not None:
        scores = scores.masked_fill(mask == 1, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    return weights @ V, weights

mask_t = torch.triu(torch.ones(seq_len, seq_len), diagonal=1)
out_manual, w_manual = attention_torch(X_t, X_t, X_t, mask=mask_t)
print(f"  Manual torch attention output shape: {tuple(out_manual.shape)}")
print(f"  Manual weights row 0 (token 'le'): {np.round(w_manual[0].numpy(), 3).tolist()}")
print(f"  Expected: [1.0, 0, 0, 0, 0] — first token only sees itself.")


  PyTorch version: 2.11.0+cpu
  Output shape (no mask)     : (1, 1, 5, 512)
  Output shape (causal mask) : (1, 1, 5, 512)
  Manual torch attention output shape: (5, 512)
  Manual weights row 0 (token 'le'): [1.0, 0.0, 0.0, 0.0, 0.0]
  Expected: [1.0, 0, 0, 0, 0] — first token only sees itself.


In [11]:
Q_t.shape

torch.Size([1, 1, 5, 512])